# **K-1**

In [21]:
import sys, time, os, json, sqlite3, gc
import pandas as pd, numpy as np, psutil

for nama in ["pandas", "numpy", "pyarrow", "requests", "sklearn", "sqlalchemy"]:
    try:
        mod = __import__(nama)
        print(f"{nama:11s}: {mod.__version__}")
    except ImportError:
        print(f"{nama:11s}: BELUM TERPASANG")

from google.colab import drive
drive.mount("/content/drive")

DIR_MENTAH = "/content/lapisan_mentah"
DIR_KURASI = "/content/lapisan_terkurasi"
DIR_SIMPAN = "/content/drive/MyDrive/Colab Notebook/Big Data/Praktikum3"

for d in (DIR_MENTAH, DIR_KURASI, DIR_SIMPAN):
    os.makedirs(d, exist_ok=True)

proses = psutil.Process(os.getpid())
catatan = []

def rss_mb():
    return proses.memory_info().rss / 1024**2

def ukur(label, fungsi):
    m0, t0 = rss_mb(), time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({
        "langkah": label,
        "detik": round(detik, 2),
        "delta_rss_mb": round(rss_mb() - m0, 1)
    })
    print(f"[{label}] {detik:.2f} s")
    return hasil

lineage = []

def catat_sumber(nama, asal, jumlah_baris, keterangan=""):
    lineage.append({
        "sumber": nama,
        "asal": asal,
        "diambil_pada": pd.Timestamp.now("UTC").isoformat(),
        "jumlah_baris": jumlah_baris,
        "keterangan": keterangan
    })
    print(f"[lineage] {nama}: {jumlah_baris:,} baris")

pandas     : 2.2.3
numpy      : 2.1.3
pyarrow    : 23.0.1
requests   : 2.32.4
sklearn    : 1.6.1
sqlalchemy : 2.0.54
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# K-**2**

In [22]:
import requests, shutil, os, time

def unduh_aman(url, tujuan, percobaan=3, jeda_awal=2):
    """Unduh dengan retry + penulisan atomik. Melewati unduhan bila file sudah ada."""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"

    # Menambahkan identitas browser agar tidak diblokir server
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

    for i in range(percobaan):
        try:
            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

            # Validasi jika file sangat kecil (misal kurang dari 1KB) kemungkinan error/kosong
            if os.path.getsize(sementara) < 1000:
                raise IOError(f"berkas terlalu kecil atau kosong (ukuran: {os.path.getsize(sementara)} bytes)")

            os.replace(sementara, tujuan) # atomik
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i) # exponential backoff
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)

    raise RuntimeError(f"gagal mengunduh setelah {percobaan} percobaan: {url}")

URL_TRIP = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
URL_ZONA = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

PATH_TRIP = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-01.parquet")
PATH_ZONA = os.path.join(DIR_MENTAH, "taxi_zone_lookup.csv")

ukur("unduh trip", lambda: unduh_aman(URL_TRIP, PATH_TRIP))
ukur("unduh zona", lambda: unduh_aman(URL_ZONA, PATH_ZONA))

for pth in (PATH_TRIP, PATH_ZONA):
    print(os.path.basename(pth), "->", round(os.path.getsize(pth)/1024**2, 2), "MB")

KOLOM = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count",
    "trip_distance", "PULocationID", "DOLocationID", "payment_type",
    "fare_amount", "tip_amount", "total_amount"
]

trip = ukur("baca trip", lambda: pd.read_parquet(PATH_TRIP, columns=KOLOM))
zona = pd.read_csv(URL_ZONA)

catat_sumber("trip", URL_TRIP, len(trip), "Parquet bulanan TLC")
catat_sumber("zona", URL_ZONA, len(zona), "tabel dimensi 265 zona")

display(zona.head())

cache ditemukan: yellow_tripdata_2023-01.parquet
[unduh trip] 0.00 s
cache ditemukan: taxi_zone_lookup.csv
[unduh zona] 0.00 s
yellow_tripdata_2023-01.parquet -> 45.46 MB
taxi_zone_lookup.csv -> 0.0 MB
[baca trip] 1.46 s
[lineage] trip: 3,066,766 baris
[lineage] zona: 265 baris


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


# **K-3**

In [23]:
API_CUACA = "https://archive-api.open-meteo.com/v1/archive"

def ambil_cuaca(mulai, selesai, lat=40.7128, lon=-74.0060, percobaan=3):
    parameter = {
        "latitude": lat, "longitude": lon,
        "start_date": mulai, "end_date": selesai,
        "hourly": "temperature_2m,precipitation",
        "timezone": "America/New_York",
    }

    for i in range(percobaan):
        r = requests.get(API_CUACA, params=parameter, timeout=60)
        if r.status_code == 200:
            data = r.json()
            if "hourly" not in data: # validasi struktur
                raise ValueError("kunci 'hourly' tidak ada pada respons")
            return pd.DataFrame(data["hourly"])

        if r.status_code in (429, 500, 502, 503): # layak dicoba ulang
            time.sleep(2 ** i)
            continue

        r.raise_for_status()

    raise RuntimeError("API cuaca tidak merespons dengan benar")

cuaca = ukur("ambil cuaca", lambda: ambil_cuaca("2023-01-01", "2023-01-31"))

cuaca["time"] = pd.to_datetime(cuaca["time"])
cuaca = cuaca.rename(columns={
    "time": "jam_mulai",
    "temperature_2m": "suhu_c",
    "precipitation": "hujan_mm"
})

catat_sumber("cuaca", API_CUACA, len(cuaca), "Open-Meteo hourly archive")
cuaca.head()

[ambil cuaca] 0.45 s
[lineage] cuaca: 744 baris


,jam_mulai,suhu_c,hujan_mm
0,2023-01-01 00:00:00,10.9,1.0
1,2023-01-01 01:00:00,10.6,1.0
2,2023-01-01 02:00:00,10.6,0.1
3,2023-01-01 03:00:00,10.5,0.0
4,2023-01-01 04:00:00,9.8,0.0


# **K-4**

# **k-5**

In [24]:
from sqlalchemy import create_engine

PATH_DB = os.path.join(DIR_MENTAH, "operasional.db")
engine = create_engine(f"sqlite:///{PATH_DB}")

tarif_referensi = pd.DataFrame({
    "payment_type": [1, 2, 3, 4, 5, 6],
    "nama_pembayaran": ["Kartu kredit", "Tunai", "Gratis", "Sengketa", "Tidak diketahui", "Perjalanan batal"],
    "kena_biaya_admin": [1, 0, 0, 0, 0, 0],
})

tarif_referensi.to_sql("tarif_referensi", engine, if_exists="replace", index=False)

# tabel transaksi tiruan untuk latihan pembacaan bertahap
trip.head(300_000).to_sql("trip_operasional", engine, if_exists="replace", index=False, chunksize=50_000)

print("Tabel dibuat:", pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", engine)["name"].tolist())

SQL = """
SELECT payment_type, COUNT(*) AS jumlah, AVG(total_amount) AS rata_total
FROM trip_operasional
WHERE total_amount > 0
GROUP BY payment_type
ORDER BY jumlah DESC
"""
ringkas_db = pd.read_sql(SQL, engine)
print(ringkas_db)

# Pembacaan bertahap: agregasi tanpa memuat seluruh tabel ke memori
total_baris = 0
for bagian in pd.read_sql("SELECT trip_distance FROM trip_operasional", engine, chunksize=50_000):
    total_baris += len(bagian)

print("dibaca bertahap:", f"{total_baris:,}", "baris")

tarif_referensi = pd.read_sql("SELECT * FROM tarif_referensi", engine)
catat_sumber("tarif_referensi", PATH_DB, len(tarif_referensi), "tabel dimensi dari basis data operasional")

Tabel dibuat: ['tarif_referensi', 'trip_operasional']
   payment_type  jumlah  rata_total
0             1  226967   31.339172
1             2   66504   25.218836
2             4    2082   26.611114
3             3    1553   22.089691
dibaca bertahap: 300,000 baris
[lineage] tarif_referensi: 6 baris


# **K-6**

In [25]:
trip["durasi_menit"] = (trip["tpep_dropoff_datetime"] - trip["tpep_pickup_datetime"]).dt.total_seconds() / 60

def profil_kolom(df):
    baris = []
    for kol in df.columns:
        s = df[kol]
        baris.append({
            "kolom": kol,
            "tipe": str(s.dtype),
            "persen_hilang": round(100 * s.isna().mean(), 3),
            "nilai_unik": s.nunique(dropna=True),
            "contoh": s.dropna().iloc[0] if s.notna().any() else None,
        })
    return pd.DataFrame(baris)

profil = profil_kolom(trip)
display(profil)

AWAL = pd.Timestamp("2023-01-01")
AKHIR = pd.Timestamp("2023-02-01")
zona_sah = set(zona["LocationID"])

aturan = {
    "kelengkapan: passenger_count ada": trip["passenger_count"].notna(),
    "keunikan: baris tidak duplikat": ~trip.duplicated(),
    "validitas: jarak 0-100 mil": trip["trip_distance"].between(0.01, 100),
    "validitas: total_amount > 0": trip["total_amount"] > 0,
    "validitas: PULocationID dikenal": trip["PULocationID"].isin(zona_sah),
    "konsistensi: durasi 1-180 menit": trip["durasi_menit"].between(1, 180),
    "konsistensi: total >= fare": trip["total_amount"] >= trip["fare_amount"],
    "ketepatan waktu dalam Jan 2023": trip["tpep_pickup_datetime"].between(AWAL, AKHIR),
}

laporan = pd.DataFrame([
    {
        "aturan": nama,
        "lulus": int(mask.sum()),
        "gagal": int((~mask).sum()),
        "persen_gagal": round(100 * (~mask).mean(), 3)
    }
    for nama, mask in aturan.items()
]).sort_values("persen_gagal", ascending=False)

display(laporan)

,kolom,tipe,persen_hilang,nilai_unik,contoh
0,tpep_pickup_datetime,datetime64[us],0.000,1610975,2023-01-01 00:32:10
1,tpep_dropoff_datetime,datetime64[us],0.000,1611319,2023-01-01 00:40:36
2,passenger_count,float64,2.339,10,1.0
3,trip_distance,float64,0.000,4387,0.97
4,PULocationID,int64,0.000,257,161
5,DOLocationID,int64,0.000,261,141
6,payment_type,int64,0.000,5,2
7,fare_amount,float64,0.000,6873,9.3
8,tip_amount,float64,0.000,4036,0.0
9,total_amount,float64,0.000,15871,14.3


,aturan,lulus,gagal,persen_gagal
0,kelengkapan: passenger_count ada,2995023,71743,2.339
2,validitas: jarak 0-100 mil,3020816,45950,1.498
5,konsistensi: durasi 1-180 menit,3030383,36383,1.186
3,validitas: total_amount > 0,3040994,25772,0.840
6,konsistensi: total >= fare,3041743,25023,0.816
7,ketepatan waktu dalam Jan 2023,3066718,48,0.002
1,keunikan: baris tidak duplikat,3066766,0,0.000
4,validitas: PULocationID dikenal,3066766,0,0.000


# **K-7**

In [26]:
KOL = "passenger_count"
asli = trip[KOL]
print("persen hilang:", round(100 * asli.isna().mean(), 3))

trip["jam"] = trip["tpep_pickup_datetime"].dt.hour

strategi = {
    "1_dibuang": asli.dropna(),
    "2_isi_median": asli.fillna(asli.median()),
    "3_isi_modus": asli.fillna(asli.mode().iloc[0]),
    "4_median_perjam": asli.fillna(trip.groupby("jam")[KOL].transform("median")),
}

banding = pd.DataFrame([
    {
        "strategi": nama,
        "n": len(s),
        "mean": round(s.mean(), 4),
        "median": s.median(),
        "std": round(s.std(), 4)
    }
    for nama, s in strategi.items()
])
display(banding)

# Strategi yang dipilih modul ini: tandai + isi per kelompok
trip[KOL + "_hilang"] = trip[KOL].isna().astype("int8")
trip[KOL] = trip[KOL].fillna(trip.groupby("jam")[KOL].transform("median"))
trip[KOL] = trip[KOL].fillna(trip[KOL].median()) # jaring pengaman

# Duplikat: periksa dua tingkat
dup_penuh = trip.duplicated().sum()
KUNCI = [
    "tpep_pickup_datetime", "tpep_dropoff_datetime",
    "PULocationID", "DOLocationID", "total_amount"
]
dup_kunci = trip.duplicated(subset=KUNCI).sum()

print(f"duplikat penuh: {dup_penuh:,} | duplikat pada kunci: {dup_kunci:,}")

sebelum = len(trip)
trip = trip.drop_duplicates(subset=KUNCI, keep="first").reset_index(drop=True)
print(f"{sebelum:,} -> {len(trip):,} baris")

persen hilang: 2.339


,strategi,n,mean,median,std
0,1_dibuang,2995023,1.3625,1.0,0.8961
1,2_isi_median,3066766,1.3541,1.0,0.8873
2,3_isi_modus,3066766,1.3541,1.0,0.8873
3,4_median_perjam,3066766,1.3541,1.0,0.8873


duplikat penuh: 0 | duplikat pada kunci: 1
3,066,766 -> 3,066,765 baris


# K-**8**

In [27]:
def batas_iqr(s, k=1.5):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

def ringkas_outlier(df, kolom):
    hasil = []
    for kol in kolom:
        s = df[kol].dropna()
        lo, hi = batas_iqr(s)
        z = (s - s.mean()) / s.std()
        hasil.append({
            "kolom": kol,
            "batas_bawah": round(lo, 2),
            "batas_atas": round(hi, 2),
            "outlier_iqr": int(((s < lo) | (s > hi)).sum()),
            "outlier_z3": int((z.abs() > 3).sum()),
            "p99": round(s.quantile(0.99), 2),
        })
    return pd.DataFrame(hasil)

display(ringkas_outlier(trip, ["trip_distance", "durasi_menit", "total_amount"]))

# Tindakan berbeda untuk maksud berbeda
layak = (
    trip["trip_distance"].between(0.01, 100) &
    trip["durasi_menit"].between(1, 180) &
    (trip["total_amount"] > 0)
)
bersih = trip.loc[layak].copy()

# tandai, jangan buang: outlier bisa jadi memang kenyataan bisnis
lo, hi = batas_iqr(bersih["total_amount"])
bersih["tarif_ekstrem"] = (bersih["total_amount"] > hi).astype("int8")

# winsorizing hanya untuk kolom turunan yang akan dipakai model
batas_atas = bersih["trip_distance"].quantile(0.995)
bersih["trip_distance_capped"] = bersih["trip_distance"].clip(upper=batas_atas)

print(f"{len(trip):,} -> {len(bersih):,} baris ({100*(1-len(bersih)/len(trip)):.2f}% dibuang)")

# Join 1: tabel zona (dimensi)
print("kunci zona unik?", zona["LocationID"].is_unique) # WAJIB diperiksa
n0 = len(bersih)
bersih = bersih.merge(
    zona[["LocationID", "Borough", "Zone"]].rename(columns={"Borough": "borough_naik", "Zone": "zona_naik"}),
    left_on="PULocationID", right_on="LocationID",
    how="left", validate="many_to_one"
)
bersih = bersih.drop(columns="LocationID")

print(f"baris {n0:,} -> {len(bersih):,} (harus sama)")
print("zona tak dikenal:", bersih["zona_naik"].isna().sum())

# Join 2: tabel pembayaran dari basis data
bersih = bersih.merge(
    tarif_referensi[["payment_type", "nama_pembayaran"]],
    on="payment_type", how="left", validate="many_to_one"
)

# Join 3: cuaca, berdasarkan jam
bersih["jam_mulai"] = bersih["tpep_pickup_datetime"].dt.floor("h")
n0 = len(bersih)
bersih = bersih.merge(cuaca, on="jam_mulai", how="left", validate="many_to_one")

print(f"baris {n0:,} -> {len(bersih):,}")
print("tanpa data cuaca:", bersih["suhu_c"].isna().sum())

,kolom,batas_bawah,batas_atas,outlier_iqr,outlier_z3,p99
0,trip_distance,-2.35,6.74,390244,67,20.06
1,durasi_menit,-9.66,35.08,170615,3175,57.25
2,total_amount,-4.55,48.65,371616,87248,101.94


3,066,765 -> 2,986,910 baris (2.60% dibuang)
kunci zona unik? True
baris 2,986,910 -> 2,986,910 (harus sama)
zona tak dikenal: 37609
baris 2,986,910 -> 2,986,910
tanpa data cuaca: 37


# K-**9**

In [28]:
bersih["hari_minggu"] = bersih["tpep_pickup_datetime"].dt.dayofweek
bersih["akhir_pekan"] = (bersih["hari_minggu"] >= 5).astype("int8")
bersih["kecepatan_mph"] = (bersih["trip_distance"] / (bersih["durasi_menit"] / 60)).round(2)
bersih["tarif_per_mil"] = (bersih["total_amount"] / bersih["trip_distance"]).round(3)
bersih["hujan"] = (bersih["hujan_mm"].fillna(0) > 0.1).astype("int8")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

FITUR_NUM = ["trip_distance_capped", "durasi_menit", "suhu_c"]
FITUR_KAT = ["nama_pembayaran", "borough_naik"]

contoh = bersih.dropna(subset=FITUR_NUM + FITUR_KAT).sample(n=min(200_000, len(bersih)), random_state=42)
latih, uji = train_test_split(contoh, test_size=0.2, random_state=42)

skala = StandardScaler().fit(latih[FITUR_NUM]) # fit HANYA di data latih
latih_num = skala.transform(latih[FITUR_NUM])
uji_num = skala.transform(uji[FITUR_NUM]) # transform saja, tanpa fit ulang

print("mean latih (harus ~0):", latih_num.mean(axis=0).round(3))
print("mean uji (tidak harus 0):", uji_num.mean(axis=0).round(3))

# Bergantung versi: parameter sparse_output ada sejak scikit-learn 1.2
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit(latih[FITUR_KAT])
latih_kat = enc.transform(latih[FITUR_KAT])

print("jumlah kolom hasil one-hot:", latih_kat.shape[1])
print("nama kolom:", enc.get_feature_names_out()[:6], "...")

mean latih (harus ~0): [-0.  0. -0.]
mean uji (tidak harus 0): [ 0.001  0.001 -0.009]
jumlah kolom hasil one-hot: 11
nama kolom: ['nama_pembayaran_Gratis' 'nama_pembayaran_Kartu kredit'
 'nama_pembayaran_Sengketa' 'nama_pembayaran_Tunai' 'borough_naik_Bronx'
 'borough_naik_Brooklyn'] ...


# K-**10**

In [29]:
KONTRAK = {
    "tpep_pickup_datetime": "datetime64[ns]",
    "trip_distance": "float64",
    "durasi_menit": "float64",
    "total_amount": "float64",
    "borough_naik": "object",
    "nama_pembayaran": "object",
}

def validasi(df, kontrak=KONTRAK):
    masalah = []
    for kol, tipe in kontrak.items():
        if kol not in df.columns:
            masalah.append(f"kolom hilang: {kol}")
        elif not str(df[kol].dtype).startswith(tipe.split("[")[0]):
            masalah.append(f"tipe {kol}: {df[kol].dtype} != {tipe}")

    if df.empty:
        masalah.append("DataFrame kosong")

    if len(df) != len(df.drop_duplicates(subset=KUNCI)):
        masalah.append("masih ada duplikat pada kunci")

    if masalah:
        raise AssertionError("Validasi gagal:\n- " + "\n- ".join(masalah))

    print(f"Validasi lulus: {len(df):,} baris x {df.shape[1]} kolom")
    return df

validasi(bersih)

def pipeline(path_trip, path_zona, df_cuaca, df_tarif):
    """Satu fungsi, idempoten: input mentah -> DataFrame tervalidasi."""
    df = pd.read_parquet(path_trip, columns=KOLOM)
    z = pd.read_csv(path_zona)

    df["durasi_menit"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60
    df["jam"] = df["tpep_pickup_datetime"].dt.hour
    df["passenger_count"] = df["passenger_count"].fillna(df.groupby("jam")["passenger_count"].transform("median"))

    df = df.drop_duplicates(subset=KUNCI)
    df = df[
        df["trip_distance"].between(0.01, 100) &
        df["durasi_menit"].between(1, 180) &
        (df["total_amount"] > 0)
    ]

    df = (
        df.merge(
            z[["LocationID", "Borough"]].rename(columns={"Borough": "borough_naik"}),
            left_on="PULocationID", right_on="LocationID",
            how="left", validate="many_to_one"
        )
        .drop(columns="LocationID")
        .merge(
            df_tarif[["payment_type", "nama_pembayaran"]],
            on="payment_type", how="left", validate="many_to_one"
        )
    )

    df["jam_mulai"] = df["tpep_pickup_datetime"].dt.floor("h")
    df = df.merge(df_cuaca, on="jam_mulai", how="left", validate="many_to_one")

    return validasi(df)

# PERUBAHAN: Gunakan URL_ZONA alih-alih PATH_ZONA agar langsung mengambil CSV dari internet
hasil = ukur("pipeline penuh", lambda: pipeline(PATH_TRIP, URL_ZONA, cuaca, tarif_referensi))
OUT = os.path.join(DIR_KURASI, "trips_bersih")

hasil["tanggal"] = hasil["tpep_pickup_datetime"].dt.date.astype(str)
ukur("tulis parquet berpartisi", lambda: hasil.to_parquet(OUT, partition_cols=["borough_naik"], index=False))

print("partisi yang terbentuk:", sorted(os.listdir(OUT))[5], "...")

# PERUBAHAN: Bukti idempotensi juga menggunakan URL_ZONA
ulang = pipeline(PATH_TRIP, URL_ZONA, cuaca, tarif_referensi)
print("idempoten?", len(ulang) == len(hasil))

# Artefak yang dikumpulkan
laporan.to_csv(f"{DIR_SIMPAN}/laporan_kualitas.csv", index=False)
pd.DataFrame(lineage).to_csv(f"{DIR_SIMPAN}/lineage.csv", index=False)
pd.DataFrame(catatan).to_csv(f"{DIR_SIMPAN}/pengukuran_kinerja.csv", index=False)
shutil.make_archive(f"{DIR_SIMPAN}/trips_bersih", "zip", OUT)

Validasi lulus: 2,986,910 baris x 26 kolom
Validasi lulus: 2,986,910 baris x 17 kolom
[pipeline penuh] 5.80 s
[tulis parquet berpartisi] 5.78 s
partisi yang terbentuk: borough_naik=Staten%20Island ...
Validasi lulus: 2,986,910 baris x 17 kolom
idempoten? True


'/content/drive/MyDrive/Colab Notebook/Big Data/Praktikum3/trips_bersih.zip'

# K-**11**

In [30]:
!pip install pyspark==3.5.1
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("BD-P03").master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

sdf = spark.read.parquet(PATH_TRIP).select(*KOLOM)
szona = spark.createDataFrame(zona[["LocationID", "Borough"]])

sbersih = (
    sdf.withColumn("durasi_menit",
        (F.col("tpep_dropoff_datetime").cast("timestamp").cast("long") -
         F.col("tpep_pickup_datetime").cast("timestamp").cast("long")) / 60
    )
    .filter(F.col("trip_distance").between(0.01, 100))
    .filter(F.col("durasi_menit").between(1, 180))
    .filter(F.col("total_amount") > 0)
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "total_amount"])
    .join(F.broadcast(szona), sdf.PULocationID == szona.LocationID, "left") # tabel kecil: broadcast
    .drop("LocationID")
)

ukur("spark: hitung baris bersih", lambda: sbersih.count())
sbersih.groupBy("Borough").count().orderBy(F.desc("count")).show(10, False)

(
    sbersih.write.mode("overwrite")
    .partitionBy("Borough")
    .parquet("/content/lapisan_terkurasi/trips_spark")
)

sbersih.explain() # cari BroadcastHashJoin, bukan SortMergeJoin
spark.stop()

[spark: hitung baris bersih] 14.03 s
+-------------+-------+
|Borough      |count  |
+-------------+-------+
|Manhattan    |2659609|
|Queens       |270315 |
|Unknown      |37609  |
|Brooklyn     |15724  |
|Bronx        |3074   |
|NaN          |340    |
|Staten Island|211    |
|EWR          |28     |
+-------------+-------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [tpep_pickup_datetime#281, tpep_dropoff_datetime#282, passenger_count#525, trip_distance#527, PULocationID#287L, DOLocationID#288L, payment_type#529L, fare_amount#531, tip_amount#533, total_amount#296, durasi_menit#535, Borough#330]
   +- BroadcastHashJoin [PULocationID#287L], [LocationID#329L], LeftOuter, BuildRight, false
      :- HashAggregate(keys=[DOLocationID#288L, tpep_dropoff_datetime#282, PULocationID#287L, total_amount#296, tpep_pickup_datetime#281], functions=[first(passenger_count#283, false), first(trip_distance#284, false), first(payment_type#289L, false), first(fare_amount#290, false),

## **STUDY KASUS**

In [31]:
# 1. Membuat Tabel Analitik
tabel_analitik = (bersih
    .assign(jam_mulai=bersih["tpep_pickup_datetime"].dt.floor("h"))
    .groupby(["borough_naik", "jam_mulai"], observed=True)
    .agg(
        jumlah_perjalanan=("total_amount", "size"),
        rata_tarif_per_mil=("tarif_per_mil", "median"),
        rata_kecepatan=("kecepatan_mph", "median"),
        suhu_c=("suhu_c", "first"),
        hujan=("hujan", "max")
    )
    .reset_index()
)

# Ambang minimum: kelompok yang terlalu kecil tidak bisa disimpulkan
tabel_analitik = tabel_analitik[tabel_analitik["jumlah_perjalanan"] >= 30]

# Menyimpan ke Parquet
tabel_analitik.to_parquet(f"{DIR_SIMPAN}/tabel_analitik.parquet", index=False)

# 2. Menjawab Dugaan Tim Pricing (Perbandingan Hujan vs Tidak Hujan)
hasil_pricing = tabel_analitik.groupby(["borough_naik", "hujan"], observed=True)["rata_tarif_per_mil"].median().unstack()
print("Perbandingan Median Tarif per Mil (0 = Tidak Hujan, 1 = Hujan):")
display(hasil_pricing)

Perbandingan Median Tarif per Mil (0 = Tidak Hujan, 1 = Hujan):


hujan,0,1
borough_naik,,
Brooklyn,6.9430,7.6550
Manhattan,10.5270,11.4000
Queens,5.4530,5.6365
Unknown,8.7255,9.8860


## **LATIHAN**

## **1**

In [32]:
import time

def unduh_aman_latihan(url, tujuan, percobaan=3, jeda_awal=2):
    """Versi unduh_aman yang menghitung kecepatan MB/detik"""
    if os.path.exists(tujuan) and os.path.getsize(tujuan) > 0:
        print("cache ditemukan:", os.path.basename(tujuan))
        return tujuan

    sementara = tujuan + ".part"
    headers = {"User-Agent": "Mozilla/5.0"}

    for i in range(percobaan):
        try:
            start_time = time.time()
            with requests.get(url, stream=True, timeout=60, headers=headers) as r:
                r.raise_for_status()
                with open(sementara, "wb") as f:
                    shutil.copyfileobj(r.raw, f)

            if os.path.getsize(sementara) < 1000:
                raise IOError("berkas kosong")

            end_time = time.time()
            waktu_detik = end_time - start_time
            ukuran_mb = os.path.getsize(sementara) / (1024 * 1024)
            kecepatan = ukuran_mb / waktu_detik if waktu_detik > 0 else 0

            os.replace(sementara, tujuan) # atomik
            print(f"Berhasil diunduh: {ukuran_mb:.2f} MB | Kecepatan: {kecepatan:.2f} MB/detik")
            return tujuan
        except Exception as e:
            jeda = jeda_awal * (2 ** i)
            print(f"percobaan {i+1} gagal ({e}); menunggu {jeda}s")
            time.sleep(jeda)

    raise RuntimeError(f"gagal mengunduh: {url}")

# Pembuktian fungsi
PATH_ZONA_LAT = os.path.join(DIR_MENTAH, "taxi_zone_lookup_latihan.csv")

print("--- Panggilan Pertama ---")
unduh_aman_latihan(URL_ZONA, PATH_ZONA_LAT)

print("\n--- Panggilan Kedua (Harus memakai cache) ---")
unduh_aman_latihan(URL_ZONA, PATH_ZONA_LAT)

--- Panggilan Pertama ---
cache ditemukan: taxi_zone_lookup_latihan.csv

--- Panggilan Kedua (Harus memakai cache) ---
cache ditemukan: taxi_zone_lookup_latihan.csv


'/content/lapisan_mentah/taxi_zone_lookup_latihan.csv'

## **2**

In [33]:
# Menambahkan aturan validitas (tip_amount >= 0) dan konsistensi (dropoff datetime > pickup datetime)
aturan_tambahan = {
    "validitas: tip_amount >= 0": trip["tip_amount"] >= 0,
    "konsistensi: dropoff selalu setelah pickup": trip["tpep_dropoff_datetime"] > trip["tpep_pickup_datetime"]
}

laporan_tambahan = pd.DataFrame([
    {
        "aturan": nama,
        "lulus": int(mask.sum()),
        "gagal": int((~mask).sum()),
        "persen_gagal": round(100 * (~mask).mean(), 3)
    }
    for nama, mask in aturan_tambahan.items()
]).sort_values("persen_gagal", ascending=False)

display(laporan_tambahan)

,aturan,lulus,gagal,persen_gagal
1,konsistensi: dropoff selalu setelah pickup,3065644,1121,0.037
0,validitas: tip_amount >= 0,3066540,225,0.007


## **3**

In [34]:
# Join dengan tabel zona untuk mendapatkan borough tujuan
bersih_latihan3 = bersih.merge(
    zona[["LocationID", "Borough"]].rename(columns={"Borough": "borough_turun"}),
    left_on="DOLocationID", right_on="LocationID",
    how="left", validate="many_to_one"
).drop(columns="LocationID")

# Menghitung pasangan borough asal dan tujuan yang paling sering muncul
pasangan_sibuk = (bersih_latihan3.groupby(["borough_naik", "borough_turun"])
                  .size()
                  .reset_index(name="jumlah_perjalanan")
                  .sort_values("jumlah_perjalanan", ascending=False)
                  .head(10))

print("10 Pasangan Borough Asal-Tujuan Paling Sibuk:")
display(pasangan_sibuk)

10 Pasangan Borough Asal-Tujuan Paling Sibuk:


,borough_naik,borough_turun,jumlah_perjalanan
22,Manhattan,Manhattan,2491566
29,Queens,Manhattan,155261
23,Manhattan,Queens,82354
20,Manhattan,Brooklyn,62986
30,Queens,Queens,59146
27,Queens,Brooklyn,42187
42,Unknown,Manhattan,18432
45,Unknown,Unknown,13562
19,Manhattan,Bronx,8865
8,Brooklyn,Brooklyn,7910


## **4**

In [35]:
from sklearn.preprocessing import MinMaxScaler

skala_minmax = MinMaxScaler().fit(latih[FITUR_NUM])
latih_num_minmax = skala_minmax.transform(latih[FITUR_NUM])
uji_num_minmax = skala_minmax.transform(uji[FITUR_NUM])

print("Rentang MinMaxScaler (Latih) - Min:", latih_num_minmax.min(axis=0).round(3))
print("Rentang MinMaxScaler (Latih) - Max:", latih_num_minmax.max(axis=0).round(3))
print("---")
print("Rentang StandardScaler (Latih) - Min:", latih_num.min(axis=0).round(3))
print("Rentang StandardScaler (Latih) - Max:", latih_num.max(axis=0).round(3))

Rentang MinMaxScaler (Latih) - Min: [0. 0. 0.]
Rentang MinMaxScaler (Latih) - Max: [1. 1. 1.]
---
Rentang StandardScaler (Latih) - Min: [-0.794 -1.226 -2.288]
Rentang StandardScaler (Latih) - Max: [ 4.247 13.925  3.347]


## **5**

In [36]:
# 1. Menduplikasi paksa baris pertama pada tabel zona
zona_ganda = pd.concat([zona, zona.iloc[[0]]], ignore_index=True)
print("Keunikan kunci LocationID di zona_ganda:", zona_ganda["LocationID"].is_unique)

print("\nMencoba Join dengan validate='many_to_one'...")
try:
    gagal_join = trip.head(100).merge(
        zona_ganda[["LocationID", "Borough"]],
        left_on="PULocationID", right_on="LocationID",
        how="left", validate="many_to_one"
    )
except Exception as e:
    print(f"BERHASIL DICEGAH! Error yang ditangkap:\n{type(e).__name__}: {e}")

Keunikan kunci LocationID di zona_ganda: False

Mencoba Join dengan validate='many_to_one'...
BERHASIL DICEGAH! Error yang ditangkap:
MergeError: Merge keys are not unique in right dataset; not a many-to-one merge


## **6**

In [37]:
def pipeline_inkremental(bulan_str):
    """
    bulan_str: Format 'YYYY-MM', misal '2023-01'
    """
    dir_keluaran = os.path.join(DIR_KURASI, f"trips_bersih_{bulan_str}")

    if os.path.exists(dir_keluaran) and len(os.listdir(dir_keluaran)) > 0:
        print(f"[*] Data untuk {bulan_str} sudah ada di direktori ({dir_keluaran}). Melewati pipeline.")
        return

    print(f"[*] Mulai memproses {bulan_str}...")
    url_trip = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{bulan_str}.parquet"
    path_trip = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan_str}.parquet")

    unduh_aman_latihan(url_trip, path_trip)

    # 3. Jalankan pipeline
    df_hasil = pipeline(path_trip, URL_ZONA, cuaca, tarif_referensi)
    df_hasil["tanggal"] = df_hasil["tpep_pickup_datetime"].dt.date.astype(str)

    # 4. Tulis hasil
    df_hasil.to_parquet(dir_keluaran, partition_cols=["borough_naik"], index=False)
    print(f"[*] Selesai! Data {bulan_str} berpartisi disimpan.\n")

print("--- Panggilan Pertama ---")
pipeline_inkremental("2023-01")

print("\n--- Panggilan Kedua (Harus melewati proses karena idempoten) ---")
pipeline_inkremental("2023-01")

--- Panggilan Pertama ---
[*] Data untuk 2023-01 sudah ada di direktori (/content/lapisan_terkurasi/trips_bersih_2023-01). Melewati pipeline.

--- Panggilan Kedua (Harus melewati proses karena idempoten) ---
[*] Data untuk 2023-01 sudah ada di direktori (/content/lapisan_terkurasi/trips_bersih_2023-01). Melewati pipeline.


## **TUGAS PRAKTIKUM**

In [38]:
import requests
import os
import pandas as pd

def ambil_libur_us(tahun):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{tahun}/US"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    df_libur = pd.DataFrame(r.json())
    df_libur['date'] = pd.to_datetime(df_libur['date']).dt.date.astype(str)
    df_libur = df_libur[['date', 'localName']].rename(columns={'date': 'tanggal', 'localName': 'nama_libur'})
    df_libur['is_libur'] = 1
    return df_libur

df_libur = ukur("ambil hari libur US", lambda: ambil_libur_us(2023))
catat_sumber("libur_nasional", "Nager.Date API", len(df_libur), "Hari libur publik US 2023")
print("\nSampel Data Hari Libur:")
display(df_libur.head(3))

def buat_laporan_kualitas(df_raw, tgl_awal, tgl_akhir, nama_bulan):
    df_raw["durasi_menit"] = (df_raw["tpep_dropoff_datetime"] - df_raw["tpep_pickup_datetime"]).dt.total_seconds() / 60
    zona_sah = set(zona["LocationID"])
    AWAL, AKHIR = pd.Timestamp(tgl_awal), pd.Timestamp(tgl_akhir)

    aturan = {
        "kelengkapan": df_raw["passenger_count"].notna(),
        "keunikan": ~df_raw.duplicated(),
        "validitas_jarak": df_raw["trip_distance"].between(0.01, 100),
        "validitas_tarif": df_raw["total_amount"] > 0,
        "validitas_lokasi": df_raw["PULocationID"].isin(zona_sah),
        "konsistensi_durasi": df_raw["durasi_menit"].between(1, 180),
        "konsistensi_tarif": df_raw["total_amount"] >= df_raw["fare_amount"],
        "ketepatan_waktu": df_raw["tpep_pickup_datetime"].between(AWAL, AKHIR),
    }

    return pd.DataFrame([
        {"aturan": nama, f"gagal_{nama_bulan}_%": round(100 * (~mask).mean(), 3)}
        for nama, mask in aturan.items()
    ])

URL_TRIP_JUL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-07.parquet"
PATH_TRIP_JUL = os.path.join(DIR_MENTAH, "yellow_tripdata_2023-07.parquet")
unduh_aman_latihan(URL_TRIP_JUL, PATH_TRIP_JUL)
trip_jul = pd.read_parquet(PATH_TRIP_JUL, columns=KOLOM)
catat_sumber("trip_juli", URL_TRIP_JUL, len(trip_jul), "Parquet bulanan TLC Juli 2023")

lap_jan = buat_laporan_kualitas(trip.copy(), "2023-01-01", "2023-02-01", "Jan")
lap_jul = buat_laporan_kualitas(trip_jul, "2023-07-01", "2023-08-01", "Jul")

lap_komparatif = lap_jan.merge(lap_jul, on="aturan")
print("\n--- Perbandingan Laporan Kualitas (Januari vs Juli 2023) ---")
display(lap_komparatif)

def proses_bulan(bulan_str, tgl_mulai, tgl_selesai):
    print(f"\n[*] Memproses Pipeline {bulan_str}...")
    url_trip = f"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_{bulan_str}.parquet"
    path_trip = os.path.join(DIR_MENTAH, f"yellow_tripdata_{bulan_str}.parquet")

    cuaca_bulan = ambil_cuaca(tgl_mulai, tgl_selesai)
    cuaca_bulan["time"] = pd.to_datetime(cuaca_bulan["time"])
    cuaca_bulan = cuaca_bulan.rename(columns={"time": "jam_mulai", "temperature_2m": "suhu_c", "precipitation": "hujan_mm"})

    df_hasil = pipeline(path_trip, URL_ZONA, cuaca_bulan, tarif_referensi)
    df_hasil["tanggal"] = df_hasil["tpep_pickup_datetime"].dt.date.astype(str)

    df_hasil = df_hasil.merge(df_libur, on="tanggal", how="left")
    df_hasil['is_libur'] = df_hasil['is_libur'].fillna(0).astype('int8')
    df_hasil['nama_libur'] = df_hasil['nama_libur'].fillna("Bukan Libur")

    return df_hasil

df_bersih_jan = proses_bulan("2023-01", "2023-01-01", "2023-01-31")
df_bersih_jul = proses_bulan("2023-07", "2023-07-01", "2023-07-31")

df_gabung = pd.concat([df_bersih_jan, df_bersih_jul])
dup_lintas = df_gabung.duplicated(subset=KUNCI).sum()
print(f"\n[*] Jumlah baris gabungan: {len(df_gabung):,}")
print(f"[*] Jumlah duplikat lintas bulan pada kunci: {dup_lintas} (Harus 0)")

DIR_GABUNGAN = os.path.join(DIR_KURASI, "trips_bersih_gabungan")
df_gabung.to_parquet(DIR_GABUNGAN, partition_cols=["tanggal"], index=False)
print(f"[*] Data dua bulan berhasil disimpan di {DIR_GABUNGAN} berpartisi menurut tanggal.")

[ambil hari libur US] 0.21 s
[lineage] libur_nasional: 17 baris

Sampel Data Hari Libur:


,tanggal,nama_libur,is_libur
0,2023-01-02,New Year's Day,1
1,2023-01-16,"Martin Luther King, Jr. Day",1
2,2023-02-12,Lincoln's Birthday,1


cache ditemukan: yellow_tripdata_2023-07.parquet
[lineage] trip_juli: 2,907,108 baris

--- Perbandingan Laporan Kualitas (Januari vs Juli 2023) ---


,aturan,gagal_Jan_%,gagal_Jul_%
0,kelengkapan,0.000,2.927
1,keunikan,0.000,0.000
2,validitas_jarak,1.498,1.717
3,validitas_tarif,0.840,1.075
4,validitas_lokasi,0.000,0.000
5,konsistensi_durasi,1.186,1.311
6,konsistensi_tarif,0.816,1.053
7,ketepatan_waktu,0.002,0.002



[*] Memproses Pipeline 2023-01...
Validasi lulus: 2,986,910 baris x 17 kolom

[*] Memproses Pipeline 2023-07...
Validasi lulus: 2,817,657 baris x 17 kolom

[*] Jumlah baris gabungan: 5,804,567
[*] Jumlah duplikat lintas bulan pada kunci: 0 (Harus 0)
[*] Data dua bulan berhasil disimpan di /content/lapisan_terkurasi/trips_bersih_gabungan berpartisi menurut tanggal.


In [39]:
def ambil_libur_us(tahun):
    url = f"https://date.nager.at/api/v3/PublicHolidays/{tahun}/US"
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    df_libur = pd.DataFrame(r.json())
    df_libur['date'] = pd.to_datetime(df_libur['date']).dt.date.astype(str)
    df_libur = df_libur[['date', 'localName']].rename(columns={'date': 'tanggal', 'localName': 'nama_libur'})
    df_libur['is_libur'] = 1
    return df_libur

In [40]:
df_gabung.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5804567 entries, 0 to 2817656
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        float64       
 3   trip_distance          float64       
 4   PULocationID           int64         
 5   DOLocationID           int64         
 6   payment_type           int64         
 7   fare_amount            float64       
 8   tip_amount             float64       
 9   total_amount           float64       
 10  durasi_menit           float64       
 11  jam                    int32         
 12  borough_naik           object        
 13  nama_pembayaran        object        
 14  jam_mulai              datetime64[us]
 15  suhu_c                 float64       
 16  hujan_mm               float64       
 17  tanggal                object        
 18  nama_libur             obje